In [ ]:
import MetaTrader5 as mt5

# Initialize connection
if not mt5.initialize():
    print("Initialize failed, error code:", mt5.last_error())
    quit()

# Login to account (replace with your details)
login = 101011337  # Your MT5 account login
password = "3xVuP*Cp"
server = "ICMarkets-Demo"


if not mt5.login(login, password=password, server=server):
    print("Login failed, error code:", mt5.last_error())
    mt5.shutdown()
    quit()

print("Connected to MT5 account #{}".format(login))


Login failed, error code: (-10001, 'IPC send failed')
Connected to MT5 account #101011337


: 

In [1]:
import time
import datetime as dt
import MetaTrader5 as mt5

# ==== USER SETTINGS ====
login = 101011337  # Your MT5 account login
password = "3xVuP*Cp"
server = "ICMarkets-Demo"


SYMBOL = "BTCUSDT"        # adapt to broker's symbol
TIMEFRAME = mt5.TIMEFRAME_M1
NOTIONAL = 100.0          # USDT per trade (like in your backtest)
HOLD_MINUTES = 5          # hold position for 5 minutes
MAGIC = 234567            # magic number to identify this bot's trades

# ==== INIT MT5 ====
if not mt5.initialize():
    print("MT5 initialize failed:", mt5.last_error())
    raise SystemExit

# If you want to explicitly log in (optional if MT5 terminal already logged in)
if not mt5.login(LOGIN, password=PASSWORD, server=SERVER):
    print("Login failed:", mt5.last_error())
    mt5.shutdown()
    raise SystemExit

# Ensure symbol is available
symbol_info = mt5.symbol_info(SYMBOL)
if symbol_info is None:
    print(f"Symbol {SYMBOL} not found")
    mt5.shutdown()
    raise SystemExit

if not symbol_info.visible:
    if not mt5.symbol_select(SYMBOL, True):
        print(f"Symbol {SYMBOL} could not be selected")
        mt5.shutdown()
        raise SystemExit

print("Connected and symbol ready:", SYMBOL)

# Track open trades with planned close time
# Structure: {ticket: close_time}
open_trades = {}


def get_last_bar_time():
    """Return time of last M1 bar as datetime."""
    rates = mt5.copy_rates_from_pos(SYMBOL, TIMEFRAME, 0, 2)
    # index 0 is current forming bar, index 1 is previous closed bar. [web:19]
    if rates is None or len(rates) < 2:
        return None, None
    last_closed = rates[1]
    bar_time = dt.datetime.fromtimestamp(last_closed['time'])
    bar_close = last_closed['close']
    return bar_time, bar_close


def send_buy_order(notional):
    """Send market buy order with notional in USDT."""
    tick = mt5.symbol_info_tick(SYMBOL)
    if tick is None:
        print("No tick data for symbol")
        return None

    price = tick.ask
    if price is None or price <= 0:
        print("Invalid ask price")
        return None

    volume = notional / price  # qty = notional / entry_price (like your backtest)

    # Normalize volume to symbol's lot step. [web:6]
    symbol_info = mt5.symbol_info(SYMBOL)
    volume = round(volume / symbol_info.volume_step) * symbol_info.volume_step

    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": SYMBOL,
        "volume": volume,
        "type": mt5.ORDER_TYPE_BUY,
        "price": price,
        "deviation": 50,
        "magic": MAGIC,
        "comment": "Python M1 hold-5min bot",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": symbol_info.filling_mode,
    }

    result = mt5.order_send(request)  # main order sending function. [web:6]
    print("Order send result:", result)

    if result is None or result.retcode != mt5.TRADE_RETCODE_DONE:
        print("Order failed:", result)
        return None

    # result.order -> order ticket, result.deal -> deal ticket, result.position -> position ticket
    return result.position


def close_position(position_ticket):
    """Close a BUY position by sending a SELL order with same volume."""
    positions = mt5.positions_get(ticket=position_ticket)
    if positions is None or len(positions) == 0:
        print(f"No position found for ticket {position_ticket}")
        return False

    pos = positions[0]
    volume = pos.volume

    tick = mt5.symbol_info_tick(SYMBOL)
    if tick is None:
        print("No tick data for close")
        return False

    price = tick.bid
    symbol_info = mt5.symbol_info(SYMBOL)

    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": SYMBOL,
        "volume": volume,
        "type": mt5.ORDER_TYPE_SELL,
        "position": position_ticket,
        "price": price,
        "deviation": 50,
        "magic": MAGIC,
        "comment": "Close after 5 min",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": symbol_info.filling_mode,
    }

    result = mt5.order_send(request)
    print("Close result:", result)

    if result is None or result.retcode != mt5.TRADE_RETCODE_DONE:
        print("Close failed:", result)
        return False
    return True


def main_loop():
    print("Starting main loop")
    last_bar_time_seen = None

    try:
        while True:
            bar_time, bar_close = get_last_bar_time()
            if bar_time is None:
                time.sleep(1)
                continue

            # New closed bar -> new trade event (like your df row)
            if last_bar_time_seen is None or bar_time > last_bar_time_seen:
                print(f"New bar closed at {bar_time}, close={bar_close}")
                last_bar_time_seen = bar_time

                # 1. Open a BUY with fixed notional
                position_ticket = send_buy_order(NOTIONAL)
                if position_ticket is not None:
                    # Schedule close time = now + HOLD_MINUTES
                    close_time = dt.datetime.now() + dt.timedelta(minutes=HOLD_MINUTES)
                    open_trades[position_ticket] = close_time
                    print(f"Opened position {position_ticket}, will close at {close_time}")

            # 2. Check if any scheduled position must be closed
            now = dt.datetime.now()
            to_close = [ticket for ticket, ctime in open_trades.items() if now >= ctime]

            for ticket in to_close:
                print(f"Attempting to close position {ticket} at {now}")
                ok = close_position(ticket)
                if ok:
                    print(f"Position {ticket} closed")
                else:
                    print(f"Failed to close position {ticket}")
                open_trades.pop(ticket, None)

            time.sleep(1)  # avoid tight loop

    except KeyboardInterrupt:
        print("Stopping bot...")

    finally:
        mt5.shutdown()
        print("MT5 connection closed")


if __name__ == "__main__":
    main_loop()


NameError: name 'LOGIN' is not defined